# 39 · EP, seed 42

- 39_resizemix50_seed42 trained with Centered EP instead of backprop: K = 10 sweeps, β = 0.01, BatchNorm statistics per sweep and phase.
- Same model, data, seed, optimizer, LR schedule, label smoothing and per-tensor clip 1.0 as BP.


In [ ]:
import os
import time
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# torch.set_num_threads(8)
# print("PyTorch CPU threads:", torch.get_num_threads())

from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from src.dataset.balanced_class_subset import BalancedClassSubset
from src.dataset.tiny_imagenet_dataset import TinyImageNetLoader, get_tiny_imagenet_class_names
from archive.utils import means, stds, convert_to_rgb, show_image, show_mixed_images, get_device, print_device_info
from src.dataset.resizemix import ResizeMixDataset
from src.ep.centered_ep import CenteredEP

RESUME = False
SEED = 42
NUM_CLASSES = 200
TRAIN_PER_CLASS = 500
VAL_PER_CLASS = 50
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 100
INITIAL_LR = 0.05
MIN_LR = 1e-05
WEIGHT_DECAY = 0.0001
LABEL_SMOOTHING = 0.1
CLASSIFIER_DROPOUT = 0.35
CLEAN_EVAL_INTERVAL = 5

RESIZEMIX_SCALE = (0.1, 0.8)
RESIZEMIX_PROB = 0.50

EP_STEPS = 10  # sweeps per nudged phase (K = 7 fails the gradient check)
EP_BETA = 0.01

SAVE_DIR = Path('checkpoints_39_resizemix50_ep_seed42')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = get_device()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Device:', DEVICE)
print('DataLoader workers:', NUM_WORKERS)
print('Epochs:', EPOCHS)
print(f'ResizeMix on {RESIZEMIX_PROB:.0%} of images | scale={RESIZEMIX_SCALE} | RandAugment after mixing')
print(f'Training: Centered EP | K={EP_STEPS} sweeps | beta={EP_BETA}')

In [ ]:
print_device_info(DEVICE)

## 1. Local model and reference dataset loader

In [ ]:
TINY_ROOT = Path('../data/tiny_imagenet/tiny-imagenet-200')
SELECTED_CLASSES = list(range(NUM_CLASSES))
print('Dataset root:', TINY_ROOT)

In [ ]:
before_mix_tf = transforms.Compose(
    [transforms.Lambda(convert_to_rgb), transforms.RandomCrop(64, padding=4, padding_mode='reflect'),
     transforms.RandomHorizontalFlip(p=0.5)])
after_mix_tf = transforms.Compose(
    [transforms.RandAugment(num_ops=2, magnitude=5, interpolation=InterpolationMode.BILINEAR), transforms.ToTensor(),
     transforms.Normalize(mean=means, std=stds)])
test_tf = transforms.Compose(
    [transforms.Lambda(convert_to_rgb), transforms.ToTensor(), transforms.Normalize(mean=means, std=stds)])

train_dataset, val_dataset, _, _ = TinyImageNetLoader(seed=SEED, root=TINY_ROOT, train_transform=before_mix_tf,
                                                      test_transform=test_tf).load()
clean_train_dataset, _, _, _ = TinyImageNetLoader(seed=SEED, root=TINY_ROOT, train_transform=test_tf,
                                                  test_transform=test_tf).load()

if not hasattr(clean_train_dataset, 'transform'):
    raise AttributeError('Expected the dataset to expose a transform attribute.')

train_subset = BalancedClassSubset(train_dataset, SELECTED_CLASSES, TRAIN_PER_CLASS, seed=SEED)
clean_train_subset = BalancedClassSubset(clean_train_dataset, SELECTED_CLASSES, TRAIN_PER_CLASS, seed=SEED)
val_subset = BalancedClassSubset(val_dataset, SELECTED_CLASSES, VAL_PER_CLASS, seed=SEED + 1)

assert train_subset.samples == clean_train_subset.samples, 'Clean and augmented training subsets differ.'
assert len(train_subset) == NUM_CLASSES * TRAIN_PER_CLASS
assert len(val_subset) == NUM_CLASSES * VAL_PER_CLASS

FULL_CLASS_NAMES = get_tiny_imagenet_class_names(train_dataset, TINY_ROOT)
CLASS_NAMES = [FULL_CLASS_NAMES[i] for i in SELECTED_CLASSES]

print('Train:', len(train_subset), 'Validation:', len(val_subset))

## 2. DataLoaders (macOS / Jupyter safe)

In [ ]:
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

loader_options = {'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS, 'pin_memory': DEVICE.type == 'cuda',
                  'persistent_workers': NUM_WORKERS > 0, **({'prefetch_factor': 2} if NUM_WORKERS > 0 else {})}

train_mix_dataset = ResizeMixDataset(train_subset, after_mix_tf, scale=RESIZEMIX_SCALE, mix_prob=RESIZEMIX_PROB)
train_loader = DataLoader(train_mix_dataset, shuffle=True, generator=loader_generator, **loader_options)
val_loader = DataLoader(val_subset, shuffle=False, **loader_options)
clean_train_loader = DataLoader(clean_train_subset, shuffle=False, **loader_options)

print('Training batches:', len(train_loader))
print('Validation batches:', len(val_loader))

In [ ]:
# Sample check: look at these before training. Top row plain, other rows are real training images, with the pasted label share.
_state = torch.get_rng_state()
_to_rgb = lambda t: (t * torch.tensor(stds).view(3, 1, 1) + torch.tensor(means).view(3, 1, 1)).clamp(0, 1).permute(1, 2, 0).numpy()
from PIL import Image as _Image
_fig, _axs = plt.subplots(4, 8, figsize=(13, 7.6))
for _c in range(8):
    _axs[0, _c].imshow(_Image.open(train_dataset.samples[train_subset.samples[_c][0]][0]).convert('RGB'), interpolation='nearest')
    for _r in range(1, 4):
        _x, _, _, _lam = train_mix_dataset[_c]
        _axs[_r, _c].imshow(_to_rgb(_x), interpolation='nearest')
        _axs[_r, _c].set_title(f'{1 - _lam.item():.2f}', fontsize=8)
for _ax in _axs.flat: _ax.axis('off')
plt.tight_layout()
plt.show()
torch.set_rng_state(_state)

## Visual Check

In [ ]:
# NUM_SHOW = 5
#
# train_iter = iter(train_loader)
# clean_train_iter = iter(clean_train_loader)
# val_iter = iter(val_loader)

In [ ]:
# train_images, train_labels = next(train_iter)
# clean_train_images, clean_train_labels = next(clean_train_iter)
# val_images, val_labels_batch = next(val_iter)
#
# show_image(train_images, train_labels, NUM_SHOW, CLASS_NAMES, name="Train Images", has_norm=True)
# show_image(clean_train_images, clean_train_labels, NUM_SHOW, CLASS_NAMES, name="Clean Train Images", has_norm=True)
# show_image(val_images, val_labels_batch, NUM_SHOW, CLASS_NAMES, name="Validation Images", has_norm=True)

## 3. Model, optimizer and cosine scheduler

In [ ]:
import torch
import torch.nn as nn


class Our08CNNPoolBN_V2(nn.Module):
    """8Conv (2+2+2+2) | BN + LeakyReLU | MaxPool x3 | GAP | Dense 512→512→200."""

    @staticmethod
    def block(channels, pool_after=False, dilations=(1, 1), dropout=0.0):
        layers = []
        assert len(dilations) == len(channels) - 1

        for i, d in enumerate(dilations):
            layers.extend([
                nn.Conv2d(channels[i], channels[i + 1], kernel_size=3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(channels[i + 1]),
                nn.LeakyReLU(0.05, inplace=True)
            ])

        if pool_after: layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        if dropout > 0: layers.append(nn.Dropout2d(p=dropout))
        return nn.Sequential(*layers)

    def __init__(self, num_classes=200, block3_dropout=0.10, block4_dropout=0.15, classifier_dropout=0.25,
                 level4_dilation=1):
        super().__init__()

        # 64 → 32 → 16 → 8 → 8
        self.level1 = self.block([3, 32, 64], pool_after=True, dilations=(1, 1))
        self.level2 = self.block([64, 64, 128], pool_after=True, dilations=(1, 1))
        self.level3 = self.block([128, 128, 256], pool_after=True, dilations=(1, 1), dropout=block3_dropout)
        self.level4 = self.block([256, 256, 512], pool_after=False, dilations=(1, level4_dilation),
                                 dropout=block4_dropout)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()

        # GAP 512 → 512
        self.dense_transition = nn.Sequential(
            nn.Linear(512, 512, bias=False),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.05, inplace=True),
            nn.Dropout(p=classifier_dropout)
        )
        self.classifier = nn.Linear(512, num_classes)

        # Weight initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.05, mode="fan_out", nonlinearity="leaky_relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

        nn.init.kaiming_normal_(self.dense_transition[0].weight, a=0.05, mode="fan_in", nonlinearity="leaky_relu")
        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x, return_features=False):
        x = self.level1(x)
        x = self.level2(x)
        x = self.level3(x)
        x = self.level4(x)

        features = self.flatten(self.avgpool(x))
        embedding = self.dense_transition(features)
        logits = self.classifier(embedding)

        return (logits, embedding) if return_features else logits


# ==========================================
# MODEL INITIALIZATION
# ==========================================

base_model = Our08CNNPoolBN_V2(num_classes=NUM_CLASSES, block3_dropout=0.15, block4_dropout=0.25,
                               classifier_dropout=0.35, level4_dilation=2)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Parameters: {total_params:,} ({total_params / 1e6:.3f}M)")
print("Architecture: 8Conv (2+2+2+2) | MaxPool x3 | BatchNorm x8 | LeakyReLU 0.05 | GAP → 512 → 512 → 200")

model = base_model.to(DEVICE)


def raw_model(): return model


# LOSS, OPTIMIZER & SCHEDULER
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.SGD(model.parameters(), lr=INITIAL_LR, momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)

TOTAL_STEPS = EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_STEPS, eta_min=MIN_LR)

# SANITY CHECK
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 64, 64, device=DEVICE)
    logits, features = model(dummy, return_features=True)

print("Logits shape:", logits.shape)
print("Feature shape:", features.shape)
model.train()

In [ ]:
# Centered EP replaces backprop. The optimizer, scheduler and clip above are used unchanged on the EP gradient.
ep = CenteredEP(model, beta=EP_BETA, steps=EP_STEPS, label_smoothing=LABEL_SMOOTHING, seed=SEED)
print(f'Centered EP: K={ep.steps} sweeps per phase, beta={ep.beta}, dropout on states {sorted(ep.drop)} = {list(ep.drop.values())}')


In [ ]:
# Timing check (optional): BP vs EP step time and GPU memory on copies of the model, with random data.
# It does not touch the model, optimizer, data order or random state used by training.
import copy

_cpu_rng = torch.get_rng_state()
_mps_rng = torch.mps.get_rng_state() if DEVICE.type == 'mps' else None
_sync = (lambda: torch.mps.synchronize()) if DEVICE.type == 'mps' else (lambda: None)
_g = torch.Generator().manual_seed(0)
_x = torch.randn(BATCH_SIZE, 3, 64, 64, generator=_g).to(DEVICE)
_a = torch.randint(NUM_CLASSES, (BATCH_SIZE,), generator=_g).to(DEVICE)
_b = torch.randint(NUM_CLASSES, (BATCH_SIZE,), generator=_g).to(DEVICE)
_lam = torch.rand(BATCH_SIZE, generator=_g).to(DEVICE)


def _median_time(fn, n=12, warm=3):
    times = []
    for i in range(n + warm):
        _sync(); t0 = time.time(); fn(); _sync()
        if i >= warm: times.append(time.time() - t0)
    return float(np.median(times))


_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, reduction='none')
_bp_model = copy.deepcopy(model).train()


def _bp_step():
    _bp_model.zero_grad(set_to_none=True)
    out = _bp_model(_x)
    (_lam * _loss(out, _a) + (1 - _lam) * _loss(out, _b)).mean().backward()


_ep_model = copy.deepcopy(model).train()
_ep = CenteredEP(_ep_model, beta=EP_BETA, steps=EP_STEPS, label_smoothing=LABEL_SMOOTHING, seed=0)
_bp_s = _median_time(_bp_step)
_ep_s = _median_time(lambda: _ep.step(_x, _a, _b, _lam))
_mem = f'{torch.mps.driver_allocated_memory() / 2 ** 30:.2f} GB' if DEVICE.type == 'mps' else 'n/a'
print(f'BP step {_bp_s * 1000:.0f} ms | EP step {_ep_s * 1000:.0f} ms ({_ep_s / _bp_s:.1f}x BP) | MPS memory {_mem}')
print(f'EP training per epoch ~{_ep_s * len(train_loader) / 60:.0f} min, {EPOCHS} epochs ~{_ep_s * len(train_loader) * EPOCHS / 3600:.0f} h (+ evaluation)')
del _bp_model, _ep_model, _ep
if DEVICE.type == 'mps': torch.mps.empty_cache()
torch.set_rng_state(_cpu_rng)
if _mps_rng is not None: torch.mps.set_rng_state(_mps_rng)


## 4. Best and latest checkpoints

In [ ]:
BEST_PATH = SAVE_DIR / 'best_model.pth'
LATEST_PATH = SAVE_DIR / 'latest_checkpoint.pth'

history = {k: [] for k in
           ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'clean_train_loss', 'clean_train_acc', 'lr',
            'generalization_gap', 'train_time_s', 'epoch_time_s']}

best_acc = -1.0
best_epoch = 0
start_epoch = 1


def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                loader=loader_generator.get_state(), ep=ep.generator.get_state())


def restore_rng(s):
    random.setstate(s['python'])
    np.random.set_state(s['numpy'])
    torch.set_rng_state(s['torch'])
    if torch.cuda.is_available() and s.get('cuda') is not None:
        torch.cuda.set_rng_state_all(s['cuda'])
    loader_generator.set_state(s['loader'])
    ep.generator.set_state(s['ep'])


def save_latest(epoch):
    payload = dict(epoch=epoch, model=raw_model().state_dict(), optimizer=optimizer.state_dict(),
                   scheduler=scheduler.state_dict(), history=history, best_acc=best_acc, best_epoch=best_epoch,
                   rng=rng_state(), config=dict(num_classes=NUM_CLASSES, batch_size=BATCH_SIZE, epochs=EPOCHS,
                                                train_per_class=TRAIN_PER_CLASS, val_per_class=VAL_PER_CLASS,
                                                classifier='8conv_v2_gap_dense_fc',
                                                classifier_dropout=CLASSIFIER_DROPOUT,
                                                augmentation='resizemix_half_of_images_randaugment2m5_after',
                                                resizemix_scale=RESIZEMIX_SCALE, resizemix_prob=RESIZEMIX_PROB,
                                                method='centered_ep', ep_steps=EP_STEPS, ep_beta=EP_BETA))
    temp = LATEST_PATH.with_suffix('.tmp')
    torch.save(payload, temp)
    os.replace(temp, LATEST_PATH)


if RESUME:
    ck = torch.load(LATEST_PATH, map_location='cpu', weights_only=False)
    cfg = ck['config']

    if cfg['batch_size'] != BATCH_SIZE or cfg['epochs'] != EPOCHS or cfg.get(
            'classifier') != '8conv_v2_gap_dense_fc' or cfg.get(
            'classifier_dropout') != CLASSIFIER_DROPOUT or cfg.get(
            'augmentation') != 'resizemix_half_of_images_randaugment2m5_after' or cfg.get(
            'resizemix_scale') != RESIZEMIX_SCALE or cfg.get('resizemix_prob') != RESIZEMIX_PROB or cfg.get(
            'method') != 'centered_ep' or cfg.get('ep_steps') != EP_STEPS or cfg.get('ep_beta') != EP_BETA:
        raise ValueError(
            'Resume configuration does not match this GAP + ResizeMix + Centered EP experiment.')

    raw_model().load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler'])

    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)

    history = ck['history']
    best_acc = ck['best_acc']
    best_epoch = ck['best_epoch']
    start_epoch = ck['epoch'] + 1
    restore_rng(ck['rng'])

    print(f'Resuming from epoch {start_epoch} | LR = {scheduler.get_last_lr()[0]:.10f}')
else:
    print('Starting fresh 8Conv V2 + GAP + Dense + ResizeMix on half of images, Centered EP training from epoch 1.')

## 5. Training and validation

In [ ]:
per_image_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, reduction='none')


def run_epoch(loader, training=False):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = torch.zeros((), device=DEVICE)
    total_correct = torch.zeros((), device=DEVICE)
    total = 0
    context = torch.no_grad() if training else torch.inference_mode()  # EP needs no autograd

    with context:
        for batch in tqdm(loader, leave=False, desc='Train' if training else 'Eval'):
            images = batch[0].to(DEVICE, non_blocking=True)

            if training:
                labels_a, labels_b, lam = (item.to(DEVICE, non_blocking=True) for item in batch[1:])
                optimizer.zero_grad(set_to_none=True)

                # EP: feedforward + two nudged phases; writes the EP gradient to p.grad and updates BN running stats
                logits = ep.step(images, labels_a, labels_b, lam)
                loss = (lam * per_image_loss(logits, labels_a) + (1.0 - lam) * per_image_loss(logits, labels_b)).mean()
                predictions = logits.detach().argmax(dim=1)
                correct = (lam * (predictions == labels_a) + (1.0 - lam) * (predictions == labels_b)).sum()

                for p in model.parameters():  # per-tensor clip at 1.0, same as BP
                    torch.nn.utils.clip_grad_norm_(p, 1.0)
                optimizer.step()
                scheduler.step()
            else:
                labels = batch[1].to(DEVICE, non_blocking=True)
                logits = model(images)
                loss = criterion(logits, labels)
                correct = (logits.detach().argmax(dim=1) == labels).sum()

            batch_size = images.size(0)
            total += batch_size
            total_loss += loss.detach() * batch_size
            total_correct += correct

    avg_loss = (total_loss / total).item()
    accuracy = (total_correct * (100.0 / total)).item()

    return avg_loss, accuracy

In [ ]:
for epoch in tqdm(range(start_epoch, EPOCHS + 1)):
    epoch_start = time.time()
    train_loss, train_acc = run_epoch(train_loader, training=True)
    train_time = time.time() - epoch_start

    if epoch == 1 or epoch % CLEAN_EVAL_INTERVAL == 0:
        clean_train_loss, clean_train_acc = run_epoch(clean_train_loader, training=False)
    else:
        clean_train_loss, clean_train_acc = None, None

    val_loss, val_acc = run_epoch(val_loader, training=False)
    current_lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss,
                     'val_acc': val_acc, 'clean_train_loss': clean_train_loss, 'clean_train_acc': clean_train_acc,
                     'lr': current_lr,
                     'generalization_gap': clean_train_acc - val_acc if clean_train_acc is not None else None,
                     'train_time_s': train_time, 'epoch_time_s': epoch_time}

    for name, value in epoch_metrics.items():
        history[name].append(value)

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch
        torch.save({'model': raw_model().state_dict(), 'epoch': epoch, 'val_acc': val_acc, 'class_names': CLASS_NAMES,
                    'experiment': 'Our 08 V2: max-pool after levels 1, 2, 3, Dropout2d, BatchNorm, GAP + dense 512 head, archive flow, ResizeMix on half of images, RandAugment after mixing, Centered EP training'},
                   BEST_PATH)

    save_latest(epoch)

    clean_text = f'{clean_train_acc:.2f}% / {clean_train_loss:.4f}' if clean_train_acc is not None else 'Skipped'
    print(
        f'Epoch {epoch:03d}/{EPOCHS} | Train: {train_acc:.2f}% / {train_loss:.4f} | Val: {val_acc:.2f}% / {val_loss:.4f} | Clean Train: {clean_text} | LR: {current_lr:.7f} | Best: {best_acc:.2f}% (epoch {best_epoch}) | Time: {train_time / 60:.1f} min train, {epoch_time / 60:.1f} min total')

print(f'Training finished at epoch {EPOCHS}. Best validation accuracy: {best_acc:.2f}% (epoch {best_epoch})')

## 6. Training and validation curves

In [ ]:
h = pd.DataFrame(history)
fig, axs = plt.subplots(2, 2, figsize=(13, 9))

axs[0, 0].plot(h.epoch, h.train_acc, label='Train (EP, 50% ResizeMix)')
axs[0, 0].plot(h.epoch, h.val_acc, label='Clean val')
axs[0, 0].plot(h.epoch, h.clean_train_acc, label='Clean train (every 5 epochs)', marker='o', markersize=3)
axs[0, 0].set_title('Accuracy (%)')
axs[0, 0].legend()

axs[0, 1].plot(h.epoch, h.train_loss, label='Train')
axs[0, 1].plot(h.epoch, h.val_loss, label='Val')
axs[0, 1].plot(h.epoch, h.clean_train_loss, label='Clean train (every 5 epochs)', marker='o', markersize=3)
axs[0, 1].set_title('Loss')
axs[0, 1].legend()

axs[1, 0].plot(h.epoch, h.lr)
axs[1, 0].set_title('Learning rate')

axs[1, 1].plot(h.epoch, h.generalization_gap)
axs[1, 1].set_title('Clean-train minus clean-val accuracy')

for ax in axs.flat:
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

best = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
raw_model().load_state_dict(best['model'])
model.eval()

clean_train_loss, clean_train_acc = run_epoch(clean_train_loader)
clean_val_loss, clean_val_acc = run_epoch(val_loader)

print(
    f'Best checkpoint: clean train {clean_train_acc:.2f}%, clean val {clean_val_acc:.2f}%, gap {clean_train_acc - clean_val_acc:.2f} pp')

## 7. Object size and position

In [ ]:
# Accuracy by object size and position. Val boxes are used for this analysis only, never for training or prediction.
from torchvision.transforms.functional import gaussian_blur

raw_model().eval()
_means = torch.tensor(means, device=DEVICE).view(1, 3, 1, 1)
_stds = torch.tensor(stds, device=DEVICE).view(1, 3, 1, 1)

def val_correct(blur_sigma=None):
    out = []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if blur_sigma is not None:
                k = 3 if blur_sigma < 0.8 else 5
                x = (gaussian_blur(x * _stds + _means, [k, k], [blur_sigma, blur_sigma]) - _means) / _stds
            out.append((raw_model()(x).argmax(1) == y).cpu())
    return torch.cat(out).numpy()

_correct = val_correct()
_ann = {}
with open(TINY_ROOT / 'val' / 'val_annotations.txt') as f:
    for line in f:
        p = line.split()
        _ann[p[0]] = tuple(map(int, p[2:6]))
_box = np.array([_ann[val_dataset.samples[i][0].name] for i, _ in val_subset.samples], dtype=float)
_w, _h = _box[:, 2] - _box[:, 0], _box[:, 3] - _box[:, 1]
_area = _w * _h / 4096
_off = np.hypot((_box[:, 0] + _box[:, 2]) / 2 - 32, (_box[:, 1] + _box[:, 3]) / 2 - 32)

print(f'All images: {_correct.mean() * 100:.2f}%')
print('Object fills (33: 46.11, 55.81, 65.99, 70.98, 68.85; 38: 45.33, 55.12, 63.76, 69.19, 66.81):')
for lo, hi in [(0, 0.1), (0.1, 0.25), (0.25, 0.5), (0.5, 0.8), (0.8, 1.01)]:
    m = (_area >= lo) & (_area < hi)
    print(f'  {lo * 100:>3.0f}-{min(hi, 1) * 100:>3.0f}%  n={m.sum():>5}  acc={_correct[m].mean() * 100:.2f}%')
print('Distance from centre, nearest to farthest quarter (33: 68.64, 69.80, 64.08, 53.44; 38: 67.00, 67.20, 63.12, 52.00):')
for part in np.array_split(np.argsort(_off), 4):
    print(f'  {_off[part].min():>4.0f}-{_off[part].max():>4.0f} px  acc={_correct[part].mean() * 100:.2f}%')